# Número principal por identidade formal do tribunal

Exploração somente leitura do texto canônico. O notebook localiza primeiro o bloco formal que declara os autos do próprio acórdão e só então extrai o número. Cobertura e qualidade são reportadas separadamente; nenhuma tabela de respostas é usada.

# 1. Setup

O SQLite é aberto em modo read-only. O universo é formado pelos 1.000 acórdãos, com 200 registros de cada tribunal. Posições são medidas no texto com espaços compactados.

In [ ]:
import hashlib
import re

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from bracis_jusbrasil.database import connect_database, get_database_path

connection = connect_database(get_database_path())
TRIBUNAIS = ('STF', 'STJ', 'TSE', 'TST', 'STM')
LIMITES = (250, 500, 1000, 2500, 5000, 10000)

def frame(sql, params=()):
    return pd.read_sql_query(sql, connection, params=params)

def compactar(texto):
    return ' '.join((texto or '').split())

def preview(texto, limit=420):
    texto = compactar(texto)
    return texto[:limit] + ('…' if len(texto) > limit else '')

def normalizar_numero(value):
    return re.sub(r'[^0-9]', '', value) if value else None

acordaos = frame('''SELECT documento_id, id, tribunal, ano, relator, texto
                   FROM documentos
                   WHERE natureza = 'acordao'
                   ORDER BY documento_id''')
assert len(acordaos) == 1000
assert acordaos.groupby('tribunal').size().eq(200).all()
acordaos['texto_compacto'] = acordaos.texto.map(compactar)
print(acordaos.groupby('tribunal').size().to_string())

# 2. Definição de bloco formal de identidade

## Bloco formal de identidade

É o trecho do próprio acórdão cuja função textual é declarar qual processo ou recurso aquele documento representa. Ele combina uma âncora institucional ou de classe com o número que a identifica: RECURSO ESPECIAL Nº ..., APELAÇÃO CRIMINAL Nº ... ou Vistos, relatados e discutidos estes autos de ... nº TST-....

O número só é aceito dentro desse bloco. Sequências numéricas na ementa, precedentes citados, artigos, valores, números internos entre parênteses e processos de outro grau são contraexemplos. O bloco, sua posição e a evidência textual são preservados.

# 3. Inventário estrutural por tribunal

Antes dos parsers, agrupamos os 200 registros de cada tribunal por estrutura observável. A frequência é exata no corpus; os exemplos são textos canônicos reais, não tabela externa nem fontes externas.

In [ ]:
def inventariar_padrao(row):
    s = row.texto_compacto.upper() if row.tribunal == 'TST' else row.texto_compacto[:6000].upper()
    t = row.tribunal
    if t == 'STF' and re.search(r'(?:PRIMEIRA|SEGUNDA|TERCEIRA|QUARTA|QUINTA|SEXTA) TURMA|PLEN[ÁA]RIO', s):
        return 'órgão julgador + classe + número + estado + relator'
    if t == 'STJ' and re.search(r'\bN[ºO.]\s*\d', s[:1800]) and re.search(r'\bRELATORA?\b', s[:2200]):
        return 'classe + Nº + número do recurso + UF + relator'
    if t == 'TSE' and re.search(r'TR(?:IB|IE)UNAL SUPERIOR ELEITORAL', s) and re.search(r'AC[ÓO€]RD[ÃA]O', s):
        return 'instituição + acórdão + classe + CNJ + local'
    if t == 'TST' and re.search(r'VISTOS,? RELATADOS E DISCUTIDOS ESTES AUTOS', s):
        return 'Vistos/autos + classe recursal + identificador TST/CNJ'
    if t == 'STM' and re.search(r'\b(?:APELA[ÇC][AÃ]O|AGRAVO|EMBARGOS|HABEAS|CONFLITO|RECURSO|REPRESENTA[ÇC][AÃ]O|RECLAMA[ÇC][AÃ]O)', s[:5000]):
        return 'instituição/ata + classe + Nº + CNJ + UF'
    return 'estrutura não classificada pela inspeção'

inventario = pd.concat([
    acordaos[['tribunal', 'documento_id', 'id']],
    acordaos.apply(inventariar_padrao, axis=1).rename('padrão')
], axis=1)
inventario['exemplo'] = acordaos.texto_compacto.map(lambda x: preview(x, 260))
inventario['localização típica'] = inventario.tribunal.map({
    'STF': 'cabeçalho, antes de RELATOR',
    'STJ': 'primeiro cabeçalho, antes de RELATOR',
    'TSE': 'cabeçalho institucional, antes de Relator',
    'TST': 'bloco Vistos/autos, frequentemente tardio',
    'STM': 'ata/cabeçalho, antes de RELATOR',
})
inventario['observações'] = inventario.tribunal.map({
    'STF': 'UF é textual; cabeçalhos podem ser republicados.',
    'STJ': 'o identificador entre parênteses é secundário.',
    'TSE': 'há OCR sem pontuação CNJ segura.',
    'TST': 'a identidade não vem necessariamente da ementa.',
    'STM': 'há variação de classes com âncora Nº estável.',
})
inventario_resumo = (inventario.groupby(['tribunal', 'padrão'], as_index=False)
                     .agg(quantidade=('id', 'size'), exemplo=('exemplo', 'first'),
                          localização_típica=('localização típica', 'first'),
                          observações=('observações', 'first')))
display(inventario_resumo)

# 4. STF — parser estrutural

O delimitador é o órgão julgador (PRIMEIRA/SEGUNDA/... TURMA ou PLENÁRIO) até a primeira declaração RELATOR(A). O número pontuado é lido somente nesse cabeçalho. Um CNJ em RELATOR DO PROCESSO posterior fica fora do bloco.

# 5. STJ — parser estrutural

O bloco começa na classe recursal e termina no primeiro RELATOR(A). Nº é parte da estrutura. O número imediatamente após Nº é separado do identificador interno entre parênteses, como (2018/0116304-1), que permanece apenas como evidência.

# 6. TSE — parser estrutural

O bloco exige instituição, ACÓRDÃO, classe e CNJ antes de Relator. Outro CNJ mais adiante não basta. OCR sem estrutura CNJ validável permanece como suspicious.

# 7. TST — parser estrutural

A âncora é Vistos, relatados e discutidos estes autos. O trecho até , em que contém classe e identificador TST-...-CNJ; somente esse trecho é pesquisado. Isso cobre a identidade tardia sem escolher números da ementa ou precedentes.

# 8. STM — parser estrutural

O bloco é a declaração institucional/classe (APELAÇÃO, AGRAVO, EMBARGOS, RECURSO, REPRESENTAÇÃO, RECLAMAÇÃO etc.) seguida de Nº e CNJ, antes do relator. O escopo por classe exclui números da decisão e de feitos mencionados.

# 9. Extração experimental do bloco formal

Os cinco parsers são independentes. Eles retornam evidência estrutural, não pontuação de candidatos. Mais de uma identidade distinta no mesmo bloco vira ambiguous; bloco presente sem número validável vira suspicious; ausência do marcador vira unparsed.

In [ ]:
CNJ = re.compile(
    r'(?<!\d)\d{3,7}\s*-\s*\d{2}\s*[. ]\s*\d{4}\s*[. ]\s*\d\s*[. ]\s*\d{2}\s*[. ]\s*\d{4}(?:\s*/\s*[A-Z]{2})?(?!\d)', re.I)
STF_NUMBER = re.compile(r'(?<!\d)\d{1,3}(?:\.\d{3})+(?!\d)')
STJ_TITLE = re.compile(
    r'\b(?P<class>(?:RECURSO|AGRAVO|AGINT|HABEAS|EMBARGOS|MANDADO|AÇÃO|CONFLITO|CAUTELAR|QO|RECLAMAÇÃO|SUSPENSÃO|PETIÇÃO|EDCL)[^.;]{0,140}?)'
    r'\s+N[ºO.]\s*(?P<num>\d{1,3}(?:\.\d{3})*|\d+)\s*-\s*(?P<uf>[A-Z]{2})\b', re.I)
TST_NUMBER = re.compile(
    r'(?P<raw>(?:TST\s*-\s*)?[A-Z][A-Z0-9]*(?:\s*-\s*[A-Z][A-Z0-9]*)*\s*-\s*'
    r'\d{1,7}\s*-\s*\d{2}\s*[. ]\s*\d{4}\s*[. ]\s*\d\s*[. ]\s*\d{2}\s*[. ]\s*\d{4})', re.I)
STM_TITLE = re.compile(
    r'\b(?P<class>(?:APELA[ÇC][AÃ]O|AGRAVO|EMBARGOS|HABEAS CORPUS|CONFLITO|RECURSO|'
    r'A[ÇC][AÃ]O|CORREI[ÇC][AÃ]O|REPRESENTA[ÇC][AÃ]O|RECLAMA[ÇC][AÃ]O)[^,]{0,140}?)'
    r'\s+N[ºO.]\s*(?P<num>\d{3,7}\s*-\s*\d{2}\s*[. ]\s*\d{4}\s*[. ]\s*'
    r'\d\s*[. ]\s*\d{2}\s*[. ]\s*\d{4})(?:\s*/\s*[A-Z]{2})?', re.I)

def make_record(pattern, start=None, block=None, raw=None, number_pos=None, status='unparsed'):
    return {'identity_pattern': pattern, 'identity_block': block,
            'identity_block_start': start, 'numero_raw': raw,
            'numero_normalizado': normalizar_numero(raw),
            'numero_posicao': number_pos, 'parse_status': status}

def finish_record(pattern, text, start, end, matches, number_match, raw=None):
    block = text[start:end]
    values = {normalizar_numero(m.group(0)) for m in matches}
    if not matches:
        return make_record(pattern, start, block, status='suspicious')
    if len(values) > 1:
        return make_record(pattern, start, block, status='ambiguous')
    raw = raw or number_match.group(0)
    return make_record(pattern, start, block, raw, number_match.start(), 'extracted')

def parse_stf_primary_number(text):
    s = compactar(text)
    relator = re.search(r'\bRELATORA?\b', s[:5000], re.I)
    if not relator:
        return make_record('STF: órgão + classe + número + relator')
    anchors = list(re.finditer(r'\b(?:PRIMEIRA|SEGUNDA|TERCEIRA|QUARTA|QUINTA|SEXTA) TURMA\b|\bPLEN[ÁA]RIO\b', s[:relator.start()], re.I))
    start = anchors[-1].start() if anchors else 0
    matches = list(STF_NUMBER.finditer(s, start, relator.start()))
    return finish_record('STF: órgão + classe + número + relator', s, start,
                        min(len(s), relator.end() + 120), matches,
                        matches[-1] if matches else None)

def parse_stj_primary_number(text):
    s = compactar(text)
    matches = [m for m in STJ_TITLE.finditer(s[:2200])
               if re.search(r'\bRELATORA?\b', s[m.end():m.end() + 260], re.I)]
    if not matches:
        return make_record('STJ: classe + Nº + número + UF + relator')
    m = matches[0]
    relator = re.search(r'\bRELATORA?\b', s[m.end():m.end() + 260], re.I)
    return finish_record('STJ: classe + Nº + número + UF + relator', s, m.start(),
                        min(len(s), m.end() + relator.end() + 120), matches, m,
                        m.group('num'))

def parse_tse_primary_number(text):
    s = compactar(text)
    institution = re.search(r'TR(?:IB|IE)UNAL SUPERIOR ELEITORAL', s[:3000], re.I)
    if not institution:
        return make_record('TSE: instituição + acórdão + classe + CNJ')
    acordao = re.search(r'AC[ÓO€]RD[ÃA]O', s[institution.end():institution.end() + 1200], re.I)
    if not acordao:
        return make_record('TSE: instituição + acórdão + classe + CNJ')
    start = institution.start()
    number_start = institution.end() + acordao.end()
    relator = re.search(r'\b(?:RELATOR|REIATOR)\b', s[number_start:number_start + 1400], re.I)
    end = number_start + relator.start() if relator else min(len(s), number_start + 1200)
    matches = list(CNJ.finditer(s, number_start, end))
    return finish_record('TSE: instituição + acórdão + classe + CNJ', s, start,
                        min(len(s), end + 100), matches,
                        matches[0] if matches else None)

def parse_tst_primary_number(text):
    s = compactar(text)
    anchor = re.search(r'VISTOS,? RELATADOS E DISCUTIDOS ESTES AUTOS', s, re.I)
    if not anchor:
        return make_record('TST: Vistos/autos + identificador TST/CNJ')
    tail = s[anchor.end():anchor.end() + 1300]
    end_marker = re.search(r',\s*EM QUE\b', tail, re.I)
    end = anchor.end() + end_marker.start() if end_marker else min(len(s), anchor.end() + 1000)
    matches = list(TST_NUMBER.finditer(s, anchor.end(), end))
    return finish_record('TST: Vistos/autos + identificador TST/CNJ', s, anchor.start(),
                        min(len(s), end + 120), matches,
                        matches[0] if matches else None)

def parse_stm_primary_number(text):
    s = compactar(text)
    relator = re.search(r'\bRELATORA?\b', s[:5500], re.I)
    limite = relator.start() if relator else 5500
    matches = list(STM_TITLE.finditer(s[:limite]))
    if not matches:
        return make_record('STM: classe + Nº + CNJ + UF')
    m = matches[0]
    matches = [m]
    return finish_record('STM: classe + Nº + CNJ + UF', s, m.start(),
                        min(len(s), m.end() + 180), matches, m, m.group('num'))

PARSERS = {'STF': parse_stf_primary_number, 'STJ': parse_stj_primary_number,
           'TSE': parse_tse_primary_number, 'TST': parse_tst_primary_number,
           'STM': parse_stm_primary_number}
parsed = pd.DataFrame([PARSERS[r.tribunal](r.texto) for r in acordaos.itertuples()])
resultado = pd.concat([acordaos.reset_index(drop=True), parsed], axis=1)
resultado['identity_block_end'] = resultado.apply(
    lambda r: r.identity_block_start + len(r.identity_block)
    if isinstance(r.identity_block, str) and r.identity_block_start is not None else None, axis=1)
display(resultado[['tribunal', 'documento_id', 'identity_pattern',
                   'identity_block_start', 'numero_raw', 'numero_normalizado',
                   'parse_status']].head(10))
print(resultado.parse_status.value_counts(dropna=False).to_string())

In [ ]:
resultado_antes = resultado.copy()

TSE_GAP = r'[\s\W_]*'
TSE_INSTITUTION = re.compile(r'TR(?:IBUNAL|E3UNAL)' + TSE_GAP + r'(?:SUPER(?:IOR|POR)' + TSE_GAP + r')?' + r'ELEITOR(?:A' + TSE_GAP + r'L)', re.I)
TSE_ACORDAO = re.compile(r'AC(?:Ó|O|€)' + TSE_GAP + r'RD(?:Ã|A)?' + TSE_GAP + r'O', re.I)
TSE_RELATOR = re.compile(r'\b(?:RELATOR|REIATOR)\b', re.I)
TSE_TITULO = re.compile(r'\b(?:RECURSO|AGRAVO|EMBARGOS|AÇÃO|ACAO|PETIÇÃO|PETICAO|MANDADO|HABEAS|LISTA|REPRESENTAÇÃO|REPRESENTACAO|RECLAMAÇÃO|RECLAMACAO)', re.I)
TSE_FALLBACK_LABEL = re.compile(r'\b(?:REspe|AgR-REspe|ED-AgR-AR|AgR-RO|RECURSO ESPECIAL ELEITORAL|AGRAVO REGIMENTAL|EMBARGOS)\b', re.I)
TSE_LEGACY = re.compile(r'\bN\s*(?:[ºO‚.]|o)?\s*(?P<num>\d{1,3}(?:[.,]\d{3})+)\b', re.I)
TSE_NUMBER_DEGRADED = re.compile(r'(?<!\d)(?P<seq>\d(?:[\s.,;:_€‚ƒ„…†‡‰]*\d){0,6})\s*[-–—]\s*(?P<dv>\d(?:[\s.,;:_€‚ƒ„…†‡‰]*\d))\s*[\s.,;:_€‚ƒ„…†‡‰]*(?P<year>\d(?:[\s.,;:_€‚ƒ„…†‡‰]*\d){3})\s*[\s.,;:_€‚ƒ„…†‡‰]*(?P<segment>\d)\s*[\s.,;:_€‚ƒ„…†‡‰]*(?P<trib>\d(?:[\s.,;:_€‚ƒ„…†‡‰]*\d))\s*[\s.,;:_€‚ƒ„…†‡‰]*(?P<origin>\d(?:[\s.,;:_€‚ƒ„…†‡‰]*\d){3})(?!\d)', re.I)
TSE_CANONICAL = re.compile(r'\d{1,7}-\d{2}\.\d{4}\.\d\.\d{2}\.\d{4}(?:/[A-Z]{2})?$', re.I)

def _tse_digits(value):
    return re.sub(r'[^0-9]', '', value or '')

def _tse_cnj_value(match):
    return ''.join(_tse_digits(match.group(name)) for name in ('seq', 'dv', 'year', 'segment', 'trib', 'origin'))

def _tse_record(pattern, start=None, block=None, raw=None, number_pos=None, status='unparsed', family='unknown', source=None, marker_source='unknown'):
    return {'identity_pattern': pattern, 'identity_block': block, 'identity_block_start': start, 'numero_raw': raw, 'numero_normalizado': _tse_digits(raw), 'numero_posicao': number_pos, 'parse_status': status, 'numero_family': family, 'parse_source': source, 'marker_source': marker_source}

def parse_tse_primary_number(text):
    s = compactar(text)
    institution = TSE_INSTITUTION.search(s[:5000])
    if not institution:
        return _tse_record('TSE: instituição + acórdão + classe + CNJ')
    acordao = TSE_ACORDAO.search(s, institution.end(), min(len(s), institution.end() + 1500))
    number_start = acordao.end() if acordao else institution.end()
    relator = TSE_RELATOR.search(s, number_start, min(len(s), number_start + 2200))
    header_end = relator.start() if relator else min(len(s), number_start + 1800)
    marker_source = 'formal'
    if not re.fullmatch(r'TRIBUNAL\s+SUPERIOR\s+ELEITORAL', institution.group(), re.I):
        marker_source = 'ocr_tolerant'
    if acordao and not re.fullmatch(r'AC[ÓO]RD[ÃA]O', acordao.group(), re.I):
        marker_source = 'ocr_tolerant'
    block = s[institution.start():min(len(s), header_end + 100)]
    matches = list(TSE_NUMBER_DEGRADED.finditer(s, number_start, header_end))
    values = {_tse_cnj_value(match) for match in matches}
    if len(values) > 1:
        return _tse_record('TSE: instituição + acórdão + classe + CNJ', institution.start(), block, status='ambiguous', marker_source=marker_source)
    if matches:
        match = matches[0]
        raw = match.group(0)
        source = 'formal_header' if TSE_CANONICAL.fullmatch(raw) else 'normalized_header'
        return _tse_record('TSE: instituição + acórdão + classe + CNJ', institution.start(), block, raw, match.start(), 'extracted', 'cnj', source, marker_source)
    legacy = TSE_LEGACY.search(s, number_start, header_end)
    if legacy and TSE_TITULO.search(s[institution.end():legacy.start()]):
        return _tse_record('TSE: instituição + classe + número legado', institution.start(), block, legacy.group('num'), legacy.start(), 'extracted', 'legacy', 'formal_header', marker_source)
    for match in TSE_NUMBER_DEGRADED.finditer(s, header_end):
        context = s[max(header_end, match.start() - 350):match.start()]
        if TSE_FALLBACK_LABEL.search(context):
            return _tse_record('TSE: fallback intradocumento', institution.start(), block, match.group(0), match.start(), 'extracted', 'cnj', 'intradocument_fallback', marker_source)
    return _tse_record('TSE: instituição + acórdão + classe + CNJ', institution.start(), block, status='suspicious', marker_source=marker_source)

PARSERS = dict(PARSERS)
PARSERS['TSE'] = parse_tse_primary_number
parsed = pd.DataFrame([PARSERS[r.tribunal](r.texto) for r in acordaos.itertuples()])
resultado = pd.concat([acordaos.reset_index(drop=True), parsed], axis=1)
resultado['identity_block_end'] = resultado.apply(lambda r: r.identity_block_start + len(r.identity_block) if isinstance(r.identity_block, str) and r.identity_block_start is not None else None, axis=1)
print('Patch TSE aplicado; demais parsers preservados.')

### 9.1. Patch TSE: tolerância local a OCR e fallback intradocumento

A regra abaixo mantém a extração restrita ao bloco formal TSE. A normalização
reconstrói somente grupos de dígitos já presentes no candidato e preserva o
texto original, a forma bruta, a família e a origem da recuperação.

# 10. Extração do número principal

O resultado principal é resultado. numero_normalizado remove pontuação somente depois da localização no bloco formal. A posição do número é evidência auxiliar; ela não decide qual candidato vence.

In [ ]:
resultado['numero_posicao'] = pd.to_numeric(resultado.numero_posicao, errors='coerce')
metricas = (resultado.pivot_table(index='tribunal', columns='parse_status',
                                  values='id', aggfunc='count', fill_value=0)
            .reindex(TRIBUNAIS, fill_value=0))
for status in ('extracted', 'ambiguous', 'suspicious', 'unparsed'):
    if status not in metricas:
        metricas[status] = 0
metricas = metricas[['extracted', 'ambiguous', 'suspicious', 'unparsed']]
metricas['total'] = metricas.sum(axis=1)
metricas['cobertura_%'] = (100 * metricas.extracted / metricas.total).round(1)
display(metricas)
display(resultado.groupby(['tribunal', 'identity_pattern']).size().rename('registros').reset_index())

# 11. Amostra para auditoria independente

O notebook não julga a correção do próprio parsing. Com seed fixa, ele seleciona 20 casos extracted por tribunal, acrescenta todos os casos não extracted e exibe evidência suficiente para um validador externo classificar cada item como correct, incorrect ou indeterminate. Não existe coluna de julgamento preenchida pelo notebook.

In [ ]:
amostra_sucessos = (resultado[resultado.parse_status.eq('extracted')]
                   .groupby('tribunal', group_keys=False)
                   .sample(n=20, random_state=42)
                   .reset_index(drop=True))
amostra_falhas = resultado[resultado.parse_status.ne('extracted')].copy()
amostra_auditoria = pd.concat([amostra_sucessos, amostra_falhas], ignore_index=True)

def contexto_lado(row, lado, limite=320):
    if pd.notna(row.numero_posicao):
        pos = int(row.numero_posicao)
    elif pd.notna(row.identity_block_start):
        pos = int(row.identity_block_start)
    else:
        pos = 0
    if lado == 'anterior':
        return preview(row.texto_compacto[max(0, pos - limite):pos], limite)
    return preview(row.texto_compacto[pos:min(len(row.texto_compacto), pos + limite)], limite)

campos_auditoria = ['tribunal', 'documento_id', 'id', 'identity_pattern',
                    'identity_block', 'identity_block_start', 'numero_raw',
                    'numero_normalizado', 'parse_status']
auditoria = amostra_auditoria[campos_auditoria].copy()
auditoria['contexto_anterior'] = amostra_auditoria.apply(
    lambda r: contexto_lado(r, 'anterior'), axis=1).to_numpy()
auditoria['contexto_posterior'] = amostra_auditoria.apply(
    lambda r: contexto_lado(r, 'posterior'), axis=1).to_numpy()
display(auditoria[campos_auditoria + ['contexto_anterior', 'contexto_posterior']])
print('Amostra auditável por tribunal/status:')
display(auditoria.groupby(['tribunal', 'parse_status']).size().rename('quantidade').reset_index())

# 12. Falhas e padrões não cobertos

As falhas são agrupadas por causa observável, sem exceções por documento_id. suspicious significa bloco presente, mas formato numérico não validável; unparsed significa ausência do marcador esperado; ambiguous preserva mais de uma identidade estrutural distinta.

In [ ]:
def taxonomia_falha(row):
    if row.parse_status == 'ambiguous':
        return 'dois números distintos no mesmo bloco formal'
    if row.parse_status == 'suspicious':
        return 'bloco formal presente, mas número atípico/OCR'
    return 'marcador ou bloco formal não coberto'

falhas = resultado[resultado.parse_status.ne('extracted')].copy()
falhas['categoria'] = falhas.apply(taxonomia_falha, axis=1)
falhas['padrão reutilizável possível?'] = falhas.tribunal.map({
    'TSE': 'sim, ampliar normalização OCR após inspeção',
}).fillna('a investigar')
falhas_resumo = (falhas.groupby(['categoria', 'tribunal'], as_index=False)
                 .agg(quantidade=('id', 'size'), exemplo=('documento_id', 'first'),
                      observacao=('identity_block', lambda s: preview(s.dropna().iloc[0])
                                  if len(s.dropna()) else 'sem bloco')))
display(falhas_resumo)
display(falhas[['tribunal', 'documento_id', 'parse_status', 'categoria',
                'identity_block', 'numero_raw']])

tse_falhas = falhas[falhas.tribunal.eq('TSE')].copy()
def numeros_presentes(texto):
    encontrados = re.findall(r'(?<!\w)\d[\d./-]{1,30}\d(?!\w)', texto[:2200])
    return ', '.join(encontrados[:20]) if encontrados else 'nenhum padrão numérico reconhecível'

tse_falhas['trecho_bloco_esperado'] = [
    r.identity_block if isinstance(r.identity_block, str)
    else preview(r.texto_compacto[:900], 900)
    for r in tse_falhas.itertuples()
]
tse_falhas['números_presentes'] = tse_falhas.texto_compacto.map(numeros_presentes)
tse_falhas['motivo'] = tse_falhas.parse_status.map({
    'suspicious': 'marcador formal localizado, mas CNJ não validável com segurança',
    'unparsed': 'marcador ou bloco formal esperado não localizado',
})
tse_falhas['padrão reutilizável possível?'] = tse_falhas.parse_status.map({
    'suspicious': 'sim — normalização OCR estrutural, sem exceção por ID',
    'unparsed': 'a investigar — buscar variante estrutural recorrente',
})
display(tse_falhas[['documento_id', 'trecho_bloco_esperado', 'números_presentes',
                     'motivo', 'categoria', 'padrão reutilizável possível?']])
tse_falhas_resumo = (tse_falhas.groupby(['categoria', 'parse_status'], as_index=False)
                     .agg(quantidade=('id', 'size'), exemplo=('documento_id', 'first'),
                          padrão_reutilizável=('padrão reutilizável possível?', 'first')))
display(tse_falhas_resumo)

### 12.1. Diagnóstico dos 25 casos TSE anotados manualmente

O JSON manual é usado somente como conjunto de diagnóstico. A célula valida a cobertura dos mesmos 25 documentos, compara o estado antes/depois e prepara os casos recuperados para validação externa. Nenhuma anotação é convertida automaticamente em `correct`, `incorrect` ou `indeterminate`.

In [ ]:
from pathlib import Path
import json

anotacoes_tse_candidates = (
    Path('data/audits/tse_25_anotacoes.json'),
    Path('../data/audits/tse_25_anotacoes.json'),
)
anotacoes_tse_path = next(path for path in anotacoes_tse_candidates if path.exists())
anotacoes_tse = json.loads(anotacoes_tse_path.read_text(encoding='utf-8'))
assert len(anotacoes_tse) == 25
assert {item['tribunal'] for item in anotacoes_tse} == {'TSE'}
ids_anotados = {item['documento_id'] for item in anotacoes_tse}
ids_problematicos_antes = set(resultado_antes.loc[
    resultado_antes.tribunal.eq('TSE') & resultado_antes.parse_status.ne('extracted'),
    'documento_id'
])
assert ids_anotados == ids_problematicos_antes

antes_25 = resultado_antes[resultado_antes.documento_id.isin(ids_anotados)].set_index('documento_id')
depois_25 = resultado[resultado.documento_id.isin(ids_anotados)].set_index('documento_id')
recuperados_25 = depois_25[depois_25.parse_status.eq('extracted')].copy()
resultado_25 = (pd.DataFrame({'documento_id': sorted(ids_anotados)})
                 .set_index('documento_id')
                 .join(antes_25[['parse_status']].rename(columns={'parse_status': 'status_antes'}))
                 .join(depois_25[['parse_status', 'numero_raw', 'numero_normalizado',
                                  'numero_family', 'parse_source', 'marker_source',
                                  'identity_block']]
                       .rename(columns={'parse_status': 'status_depois'})))
resultado_25['contexto_relevante'] = depois_25.apply(
    lambda r: contexto_lado(r, 'posterior'), axis=1
)
display(resultado_25.reset_index()[['documento_id', 'status_antes', 'status_depois',
                                    'numero_raw', 'numero_normalizado', 'numero_family',
                                    'parse_source', 'marker_source', 'identity_block',
                                    'contexto_relevante']])

mecanismos_25 = (recuperados_25.groupby(['parse_source', 'numero_family', 'marker_source'], dropna=False)
                 .size().rename('quantidade').reset_index())
display(mecanismos_25)
mecanismos_25_resumo = pd.DataFrame({
    'mecanismo': ['normalização do candidato', 'marcador OCR tolerado',
                  'fallback intradocumento', 'número legado'],
    'quantidade': [
        int((recuperados_25.parse_source == 'normalized_header').sum()),
        int((recuperados_25.marker_source == 'ocr_tolerant').sum()),
        int((recuperados_25.parse_source == 'intradocument_fallback').sum()),
        int((recuperados_25.numero_family == 'legacy').sum()),
    ],
})
display(mecanismos_25_resumo)
print('Nota: mecanismos podem se sobrepor no mesmo caso.')
print('Casos TSE anotados:', len(resultado_25))
print('Recuperados:', len(recuperados_25))
print('Ainda suspicious/unparsed:', int((depois_25.parse_status.ne('extracted')).sum()))

antigos_175 = resultado_antes[(resultado_antes.tribunal.eq('TSE')) &
                              resultado_antes.parse_status.eq('extracted')]
depois_175 = resultado[resultado.documento_id.isin(set(antigos_175.documento_id))]
regressao_175 = antigos_175[['documento_id', 'numero_normalizado']].merge(
    depois_175[['documento_id', 'numero_normalizado', 'parse_status']],
    on='documento_id', how='left', suffixes=('_antes', '_depois')
)
regressao_175['alterado'] = (
    regressao_175.numero_normalizado_antes != regressao_175.numero_normalizado_depois
) | regressao_175.parse_status.ne('extracted')
regressao_resumo = pd.DataFrame({
    'casos_estáveis': [int((~regressao_175.alterado).sum())],
    'casos_alterados': [int(regressao_175.alterado.sum())],
})
display(regressao_resumo)
if regressao_175.alterado.any():
    display(regressao_175[regressao_175.alterado])
else:
    print('Nenhuma regressão nos 175 casos TSE previamente extracted.')

print('Casos novos preparados para validação externa:', len(recuperados_25))

# 13. Métricas consolidadas

Cobertura é o percentual de registros com número extraído. Qualidade é medida somente na amostra auditada e não é confundida com cobertura.

In [ ]:
resumo_final = metricas[['total', 'extracted', 'ambiguous', 'suspicious',
                             'unparsed', 'cobertura_%']].copy()
resumo_final['auditoria_externa'] = 'amostra preparada; não executada pelo notebook'
display(resumo_final)

def maturidade(row):
    if row['cobertura_%'] >= 95 and row['ambiguous'] == 0 and row['suspicious'] == 0 and row['unparsed'] == 0:
        return 'RESOLVIDO'
    if row['cobertura_%'] >= 70:
        return 'PARCIAL'
    return 'NÃO RESOLVIDO'

maturidade_tribunal = resumo_final.apply(maturidade, axis=1).rename('status').to_frame()
maturidade_tribunal['evidência'] = resumo_final.apply(
    lambda r: f"cobertura estrutural {r['cobertura_%']}%; auditoria externa pendente; {int(r['suspicious']) + int(r['unparsed'])} falhas não extraídas",
    axis=1)
display(maturidade_tribunal)
TRIBUNAIS_MADUROS = list(maturidade_tribunal.index[maturidade_tribunal.status.eq('RESOLVIDO')])
print('Tribunais estruturalmente completos; colisões seguem preliminares:', TRIBUNAIS_MADUROS)

# 14. Plots

Cada gráfico responde a uma pergunta distinta e mantém a distribuição por tribunal visível. Posições servem para visualizar localização, não para selecionar o número.

<!-- Seções legadas preservadas apenas como âncoras de compatibilidade: # 4. Parser estrutural do STF; # 13. Construir chave primária experimental; # 22. Findings. -->

### Cobertura TSE antes vs depois

A cobertura compara os três estados do parser nos 200 registros TSE. A leitura é descritiva: o ganho de cobertura não substitui a validação independente dos números recuperados.

In [ ]:
cobertura_tse_antes_depois = pd.DataFrame({
    'antes': resultado_antes[resultado_antes.tribunal.eq('TSE')].parse_status
        .value_counts().reindex(['extracted', 'suspicious', 'unparsed'], fill_value=0),
    'depois': resultado[resultado.tribunal.eq('TSE')].parse_status
        .value_counts().reindex(['extracted', 'suspicious', 'unparsed'], fill_value=0),
})
ax = cobertura_tse_antes_depois.T.plot.bar(rot=0)
ax.set_title('Cobertura TSE antes vs depois')
ax.set_xlabel('versão do parser')
ax.set_ylabel('acórdãos')
ax.legend(title='status')
fig = ax.get_figure()
fig.tight_layout()
plt.show()
display(pd.DataFrame({'interpretação': [
    'As regras TSE recuperam os casos diagnosticados; a amostra recuperada permanece preparada para validação externa.'
]}))

In [ ]:
# Plot 1 — posição do bloco formal por tribunal
fig, ax = plt.subplots(figsize=(8, 4))
resultado.boxplot(column='identity_block_start', by='tribunal', ax=ax)
plt.suptitle('')
ax.set_title('Posição do bloco formal de identidade por tribunal')
ax.set_xlabel('tribunal')
ax.set_ylabel('posição do bloco (caracteres compactados)')
fig.tight_layout()
plt.show()
display(pd.DataFrame({'interpretação': ['O TST tende a localizar o bloco formal mais tarde; os demais concentram a identidade no cabeçalho.']}))

In [ ]:
# Plot 2 — posição do número principal por tribunal
fig, ax = plt.subplots(figsize=(8, 4))
resultado.dropna(subset=['numero_posicao']).boxplot(column='numero_posicao', by='tribunal', ax=ax)
plt.suptitle('')
ax.set_title('Distribuição da posição do número principal por tribunal')
ax.set_xlabel('tribunal')
ax.set_ylabel('posição do número (caracteres compactados)')
fig.tight_layout()
plt.show()
display(pd.DataFrame({'interpretação': ['A posição varia dentro do bloco formal e não é usada como critério de escolha.']}))

In [ ]:
# Plot 3 — cobertura acumulada de blocos e números
linhas = []
for tribunal, grupo in resultado.groupby('tribunal'):
    for limite in LIMITES:
        linhas.append({'tribunal': tribunal, 'limite': limite, 'objeto': 'bloco',
                       'cobertura': 100 * (grupo.identity_block_start.notna() &
                                           grupo.identity_block_start.le(limite)).mean()})
        linhas.append({'tribunal': tribunal, 'limite': limite, 'objeto': 'número',
                       'cobertura': 100 * (grupo.numero_posicao.notna() &
                                           grupo.numero_posicao.le(limite)).mean()})
acumulada = pd.DataFrame(linhas)
fig, ax = plt.subplots(figsize=(9, 4))
for (tribunal, objeto), grupo in acumulada.groupby(['tribunal', 'objeto']):
    ax.plot(grupo.limite, grupo.cobertura, marker='o', label=f'{tribunal} — {objeto}')
ax.set_title('Cobertura acumulada por posição — Cobertura acumulada por janela')
ax.set_xlabel('posição limite (250, 500, 1.000, 2.500, 5.000, 10.000)')
ax.set_ylabel('% dos acórdãos')
ax.legend(ncol=2)
fig.tight_layout()
plt.show()
display(acumulada)
display(pd.DataFrame({'interpretação': ['A curva separa descoberta do bloco e extração do número, evidenciando identidades tardias.']}))

In [ ]:
# Plot 4 — status do parser
fig, ax = plt.subplots(figsize=(8, 4))
metricas[['extracted', 'ambiguous', 'suspicious', 'unparsed']].plot.bar(stacked=True, ax=ax, rot=0)
ax.set_title('Status do parsing por tribunal')
ax.set_xlabel('tribunal')
ax.set_ylabel('acórdãos')
fig.tight_layout()
plt.show()
display(pd.DataFrame({'interpretação': ['OCR, ambiguidade e ausência permanecem visíveis e não são escondidos em extraídos.']}))

In [ ]:
# Plot 5 — casos selecionados para auditoria por tribunal e parse_status
contagem_auditoria = (auditoria.groupby(['tribunal', 'parse_status']).size()
                      .unstack(fill_value=0).reindex(TRIBUNAIS, fill_value=0))
fig, ax = plt.subplots(figsize=(8, 4))
contagem_auditoria.plot.bar(stacked=True, ax=ax, rot=0)
ax.set_title('Casos selecionados para auditoria por tribunal e parse_status')
ax.set_xlabel('tribunal')
ax.set_ylabel('casos na amostra auditável')
fig.tight_layout()
plt.show()
display(pd.DataFrame({'interpretação': ['A figura mostra o que será revisado externamente, sem atribuir qualidade automaticamente.']}))

# 15. Análise preliminar de tribunal + número (condicionada à maturidade)

A etapa só roda para tribunais RESOLVIDO. Nos demais, a análise de colisões é adiada porque o número principal ainda não está maduro, evitando contaminar grupos com números contextuais.

In [ ]:
elegiveis = resultado[resultado.parse_status.eq('extracted') &
                       resultado.tribunal.isin(TRIBUNAIS_MADUROS)].copy()
if not TRIBUNAIS_MADUROS:
    print('Análise de colisões adiada porque o número principal ainda não está maduro.')
else:
    elegiveis['case_key'] = elegiveis.tribunal + '|' + elegiveis.numero_normalizado
    tamanhos = elegiveis.groupby('case_key').id.nunique()
    grupos = tamanhos.rename('ids').reset_index()
    multi = grupos[grupos.ids > 1]
    print({'tribunais': TRIBUNAIS_MADUROS, 'registros elegíveis': len(elegiveis),
           'grupos únicos': len(grupos), 'grupos multi-ID': len(multi),
           'maior grupo': int(tamanhos.max()) if len(tamanhos) else 0})
    tamanho_plot = tamanhos.map(lambda n: '1 ID' if n == 1 else ('2 IDs' if n == 2 else ('3 IDs' if n == 3 else '4+ IDs'))).value_counts()
    fig, ax = plt.subplots(figsize=(7, 4))
    tamanho_plot.reindex(['1 ID', '2 IDs', '3 IDs', '4+ IDs'], fill_value=0).plot.bar(ax=ax, rot=0)
    ax.set_title('Quantidade de grupos por tamanho')
    ax.set_xlabel('tamanho do grupo')
    ax.set_ylabel('grupos')
    fig.tight_layout()
    plt.show()
    por_tribunal = (multi.merge(elegiveis[['case_key', 'tribunal']].drop_duplicates())
                    .groupby('tribunal').size().reindex(TRIBUNAIS, fill_value=0))
    fig, ax = plt.subplots(figsize=(7, 4))
    por_tribunal.plot.bar(ax=ax, rot=0)
    ax.set_title('Grupos multi-ID por tribunal')
    ax.set_xlabel('tribunal')
    ax.set_ylabel('grupos')
    fig.tight_layout()
    plt.show()
    elegiveis['texto_hash'] = elegiveis.texto.map(lambda x: hashlib.sha256(x.encode()).hexdigest())
    colisoes = elegiveis[elegiveis.case_key.isin(multi.case_key)].copy()
    inspecao = (colisoes.groupby('case_key')
                .agg(tribunal=('tribunal', 'first'), numero=('numero_raw', 'first'),
                     ids=('id', lambda x: ', '.join(x.astype(str))),
                     textos=('texto_hash', 'nunique')).reset_index())
    inspecao['categoria'] = inspecao.textos.map(lambda n: 'A — texto idêntico' if n == 1 else 'E — indeterminado')
    display(inspecao.head(30))
    fig, ax = plt.subplots(figsize=(7, 4))
    inspecao.categoria.value_counts().plot.bar(ax=ax, rot=15)
    ax.set_title('Distribuição das categorias de colisão')
    ax.set_xlabel('categoria')
    ax.set_ylabel('grupos')
    fig.tight_layout()
    plt.show()
    comparacao = pd.DataFrame({'estratégia': ['tribunal + número principal'],
                               'cobertura': [len(elegiveis)], 'grupos multi-ID': [len(multi)]})
    display(comparacao)
    fig, ax = plt.subplots(figsize=(7, 4))
    comparacao.plot.bar(x='estratégia', y='cobertura', ax=ax, legend=False)
    ax.set_title('Cobertura versus colisões por estratégia')
    ax.set_xlabel('estratégia')
    ax.set_ylabel('registros elegíveis')
    fig.tight_layout()
    plt.show()

# 16. Findings

## Regras adicionadas

- tolerância local a marcadores TSE degradados por OCR;
- reconstrução determinística de grupos de dígitos do CNJ, sem inventar dígitos;
- suporte a separadores ausentes, duplicados ou convertidos em espaços;
- fallback intradocumento condicionado a rótulo processual estruturado;
- suporte a número curto/legado TSE;
- campos exploratórios `numero_family`, `parse_source` e `marker_source`.

## TSE antes/depois

O diagnóstico manual cobre exatamente os 25 casos antes classificados como `suspicious` ou `unparsed`. A cobertura final é exibida na tabela e no plot anteriores; o ganho não é tratado como prova automática de correção.

## Casos recuperados

Os 25 casos anotados são preparados individualmente para validação externa, com bloco formal, forma bruta, número normalizado, família, origem e marcador estrutural.

## Regressões

A comparação obrigatória dos 175 casos TSE previamente `extracted` é exibida na seção de diagnóstico. Qualquer caso alterado deve ser tratado como blocker.

## Casos restantes

A tabela de falhas permanece válida mesmo quando vazia; `suspicious` e `unparsed` continuam visíveis caso alguma regra não seja suficiente.

## Escopo

O parser estrutural dos outros tribunais e a análise de colisões não foram alterados. O SQLite permanece read-only. Não há hardcodes por `documento_id` ou `id`, nem implementação de `CanonicalIndex`.

## Próximo passo

Se não houver regressão, o parser TSE está suficientemente maduro para congelar a exploração do `.db`; a validação independente dos casos recuperados continua sendo a etapa de controle.

In [ ]:
print('Status por tribunal:')
display(resumo_final)
print('Maturidade estrutural:')
display(maturidade_tribunal)
print('O notebook agora está preparado para auditoria independente do número principal.')
connection.close()
print('Conexão read-only encerrada.')